In [ ]:
from google.colab import userdata
userdata.get('OPENAI_API_TOKEN')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
BASE_DIR="/content/drive/MyDrive/Databricks_genai_hackathon_jan2024"
DATA_FILE_PATH=f"{BASE_DIR}/datasets/WMT_Grocery_202209.csv"
VECTORDB_PATH=f"{BASE_DIR}/chromadb1"
EMBEDDING_MODEL_PATH=f"{BASE_DIR}/embedding_model"

# Installs

In [ ]:
!pip install --upgrade --quiet langchain langchain-community langchainhub langchain-openai openai transformers chromadb gradio tiktoken sentence-transformers

In [ ]:
%pip install --upgrade --quiet  langchain langsmith langchainhub --quiet

# Index

In [ ]:
from langsmith import Client

client = Client(api_key=userdata.get('LANGCHAIN_API_KEY'))

In [ ]:
import pandas
from langchain_openai import ChatOpenAI
from langchain import hub
from langchain_core.tracers.context import tracing_v2_enabled
from langchain_community.vectorstores import Chroma
from langchain_core.runnables import RunnablePassthrough
from langchain.embeddings import OpenAIEmbeddings
from langchain.document_loaders import DataFrameLoader
from langchain.embeddings import HuggingFaceEmbeddings
from sentence_transformers import SentenceTransformer
from langchain_core.output_parsers import StrOutputParser
from langchain.prompts import ChatPromptTemplate, HumanMessagePromptTemplate
from langchain_core.messages import SystemMessage

In [ ]:
# Read the product data
df = pandas.read_csv(DATA_FILE_PATH)
df = df[["index", "PRODUCT_NAME"]].drop_duplicates(subset=["PRODUCT_NAME"]).copy().head(100)
df.rename(columns={'index': 'id', 'PRODUCT_NAME': 'text'}, inplace=True)
print(df.shape)
df.head()

In [ ]:
!rm -rf /content/drive/MyDrive/Databricks_genai_hackathon_jan2024/chromadb1
!mkdir -p /content/drive/MyDrive/Databricks_genai_hackathon_jan2024/chromadb1

In [ ]:
!rm -rf /content/drive/MyDrive/Databricks_genai_hackathon_jan2024/embedding_model
!mkdir -p /content/drive/MyDrive/Databricks_genai_hackathon_jan2024/embedding_model

In [ ]:
# Create the vectordb

# Download embeddings model
original_model = SentenceTransformer('all-MiniLM-L12-v2')

# Reload model using langchain wrapper
original_model.save(EMBEDDING_MODEL_PATH)
embedding_model = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_PATH)
# embedding_model = OpenAIEmbeddings(openai_api_key=userdata.get('OPENAI_API_TOKEN'))

# Loading product dataframe
documents = DataFrameLoader(df, page_content_column="text").load()

# Create the vector db
vectordb = Chroma.from_documents(
    documents=documents, embedding=embedding_model, persist_directory=VECTORDB_PATH
)

vectordb.persist()

# Simple Bot

A simple RAG bot

In [ ]:
retriever = vectordb.as_retriever(search_kwargs={"k":10})
prompt = hub.pull("rlm/rag-prompt")
llm = ChatOpenAI(model_name="gpt-3.5-turbo-1106", temperature=0, openai_api_key=userdata.get('OPENAI_API_TOKEN'))

In [ ]:
retriever.get_relevant_documents("Hummus")

In [ ]:
prompt

In [ ]:
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
df.head()

In [ ]:
rag_chain.invoke("List all the available hummus items?")

In [ ]:
df.query("text.str.contains('Hummus')")

In [ ]:
from langchain_core.prompts import PromptTemplate

template = """Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
Use all the relevant findings from the context in your answer. Once you find the answer, just say the product names.

{context}

Question: {question}

Helpful Answer:"""

custom_rag_prompt = PromptTemplate.from_template(template)

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | custom_rag_prompt
    | llm
    | StrOutputParser()
)

print(rag_chain.invoke("Hummus"))

In [ ]:
# To Dos:

# 1. Experiment with the prompts to have llm return all the options
# 2. How to make llm behave like an order fulfilling agent?
# 2.1. Define the desired flow in your prompt? e.g. "greeting" -> "receive the order" -> "show the options (if needed)" -> "clarify on options (how to do it when there are multiple products each with different options?)" -> "follow up (i.e. confirming the basket)" -> "Finalize (e.g. do you need anything else?)" -> "Create the basket": FINISH

# 3. Optimizations:
    # 3.1. Latency
    # 3.2. LLM model choice
    # 3.3. Vector DB  choice
    # 3.4. Embedding model choice
    # 3.5. Prompt experiments -> which prompts give the best answers
    # 3.6. Document loading -> i.e. chunking
    # 3.7. What's the best way to index tabular data?

# 4. Implementation
# 4.1. Databricks tools (once we get the access)
# 4.2. LlamaIndex